# 02 — Conditional Routing

Routing chooses an execution path from state. This lesson first builds a deterministic classifier and router, then shows how an optional LLM classifier can compute the same structured category without taking ownership of policy.

## Topology

```mermaid
flowchart LR
    accTitle: Topic Routing Graph
    accDescr: A question is classified and routed to weather, trade, or politics research before all branches end.

    start([START]) --> classify[Classify]
    classify --> router{Router}
    router -->|weather| weather[Weather]
    router -->|trade| trade[Trade]
    router -->|politics| politics[Politics]
    weather --> finish([END])
    trade --> finish
    politics --> finish
```

The classifier is a node: it computes `category`. The router is control: it validates `category` and returns a branch name.

## State and deterministic classifier

Keyword classification is intentionally simple. It makes route behavior predictable while the topology is being learned.

In [1]:
import operator
from pathlib import Path
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph

from graph_engineering.control import classify_keywords, route_category
from graph_engineering.llm import DeepSeekSettings, chat, create_deepseek_client


class RoutingState(TypedDict, total=False):
    question: str
    category: str
    evidence: list[str]
    trace: Annotated[list[str], operator.add]


def deterministic_classify(state: RoutingState) -> dict:
    category = classify_keywords(state["question"])
    return {"category": category, "trace": [f"classify:{category}"]}

## Branch nodes compute domain-specific work

These deterministic branches stand in for different tools, retrievers, or agents. A node is not synonymous with an agent.

In [2]:
def weather(_: RoutingState) -> dict:
    return {"evidence": ["Weather branch selected."], "trace": ["weather"]}


def trade(_: RoutingState) -> dict:
    return {"evidence": ["Trade branch selected."], "trace": ["trade"]}


def politics(_: RoutingState) -> dict:
    return {"evidence": ["Politics branch selected."], "trace": ["politics"]}

## Build once around a classifier contract

The graph accepts any classifier node that returns a validated category update. This lets computation change without changing the route table.

In [3]:
def build_routing_graph(classifier):
    builder = StateGraph(RoutingState)
    builder.add_node("classify", classifier)
    builder.add_node("weather", weather)
    builder.add_node("trade", trade)
    builder.add_node("politics", politics)

    builder.add_edge(START, "classify")
    builder.add_conditional_edges(
        "classify",
        route_category,
        {"weather": "weather", "trade": "trade", "politics": "politics"},
    )
    builder.add_edge("weather", END)
    builder.add_edge("trade", END)
    builder.add_edge("politics", END)
    return builder.compile()


graph = build_routing_graph(deterministic_classify)

In [4]:
cases = {
    "Will a heatwave increase cooling demand?": "weather",
    "How could a tariff affect imports?": "trade",
    "What did the government announce?": "politics",
}

for question, expected in cases.items():
    result = graph.invoke({"question": question, "trace": []})
    assert result["category"] == expected
    print(result["trace"])

['classify:weather', 'weather']
['classify:trade', 'trade']
['classify:politics', 'politics']


## Optional semantic classification

Natural language categories may require semantic judgment. A safe structure is `LLM classifier → structured category → validation → deterministic router`. Provider initialization stays outside graph concepts. If no key is configured, this section prints a clear skip message and the rest of the notebook remains runnable.

In [5]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


settings = DeepSeekSettings.from_env(find_project_root() / ".env")
client = create_deepseek_client(settings)
print(
    f"DeepSeek ready with model {settings.model}."
    if client is not None
    else "Optional semantic route skipped: DEEPSEEK_API_KEY is not configured."
)

DeepSeek ready with model deepseek-v4-flash.


In [6]:
def semantic_classify(state: RoutingState) -> dict:
    if client is None:
        raise RuntimeError("Configure DEEPSEEK_API_KEY before invoking semantic routing.")

    data = chat(
        client,
        settings,
        system_prompt=(
            "You are a routing classifier. Return one JSON object only, "
            "with exactly one key named category."
        ),
        user_prompt=(
            "Classify the question into exactly one allowed category: "
            "weather, trade, or politics.\n"
            "Return exactly this JSON shape and no additional keys: "
            "{\"category\": \"weather\"}\n"
            f"Question: {state['question']}"
        ),
        json_mode=True,
        max_tokens=100,
    )
    raw_category = data.get("category") if isinstance(data, dict) else None
    if not isinstance(raw_category, str):
        raise ValueError(
            "Semantic classifier returned an invalid payload. Expected "
            f"{{'category': '<weather|trade|politics>'}}, received: {data!r}"
        )

    category = raw_category.casefold().strip()
    route_category({"category": category})  # validate the allowed value
    return {"category": category, "trace": [f"semantic_classify:{category}"]}


if client is None:
    print("No semantic graph invoked; deterministic examples above are complete.")
else:
    semantic_graph = build_routing_graph(semantic_classify)
    semantic_result = semantic_graph.invoke(
        {"question": "Will extreme heat change AC sales?", "trace": []}
    )
    assert semantic_result["category"] == "weather"
    print("Category:", semantic_result["category"])
    print("Evidence:", semantic_result["evidence"])
    print("Trace:", semantic_result["trace"])

Category: weather
Evidence: ['Weather branch selected.']
Trace: ['semantic_classify:weather', 'weather']


## Keep hard policy deterministic

LLMs can help interpret meaning. They should not normally decide whether to exceed a retry count, budget, deadline, permission, or hard safety constraint. Those values are exact and belong in code unless a deliberate, reviewed design says otherwise.

Next: Notebook 03 turns a route back into a bounded feedback loop.